# Cost-constrained time-to-scale frontier

This notebook extends the repository's time-to-scale analysis by adding a cost-effectiveness constraint. It reproduces the 50% task-completion time horizon for each model and then computes the effective frontier
that requires humans to be both faster *and* cheaper than the AI system.

The high-level workflow is:
- load the aggregated run data used across the repository;
- fit the logistic models that determine the 50% time horizon per agent;
- apply a simple AI cost decay trend and human wage assumption;
- plot the baseline and cost-constrained frontiers over time.


In [ ]:
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

from src.wrangle.logistic import run_logistic_regressions

plt.style.use("seaborn-v0_8-darkgrid")


In [ ]:
runs_path = pathlib.Path("data/external/all_runs.jsonl")
release_dates_path = pathlib.Path("data/external/release_dates.yaml")

runs = pd.read_json(runs_path, lines=True)
with open("fig_params/figs.yaml") as f:
    params = yaml.safe_load(f)

wrangle_params = params["figs"]["wrangle_logistic"]["headline"]

logistic_df = run_logistic_regressions(
    runs,
    release_dates_path,
    wrangle_params,
)

logistic_df = logistic_df[logistic_df["agent"] != "human"].copy()
logistic_df = logistic_df[logistic_df["release_date"].notna()].copy()
logistic_df["release_date"] = pd.to_datetime(logistic_df["release_date"])
logistic_df.sort_values("release_date", inplace=True)

logistic_df["p50_minutes"] = logistic_df["p50"]
logistic_df["human_time_seconds"] = logistic_df["p50_minutes"] * 60

logistic_df[["agent", "release_date", "p50_minutes"]]


### Cost and wage assumptions

The effective frontier is the minimum of the baseline 50% time horizon and the human-equivalent time implied by the AI's inference cost.
The toy cost model used here assumes:

- AI inference cost halves every 18 months;
- the earliest model starts at $120 per task;
- humans earn $50 per hour.

Under these assumptions the cutoff time at which humans remain cheaper is `AI_cost / Wage_per_second`, so the plotted effective frontier is
`min(human_time, AI_cost / Wage_per_second)` for each release date.


In [ ]:
HALVING_PERIOD_MONTHS = 18
INITIAL_INFERENCE_COST = 120.0  # dollars per task at the baseline date
HOURLY_WAGE = 50.0  # dollars per human hour

wage_per_second = HOURLY_WAGE / 3600
baseline_date = logistic_df["release_date"].min()

def months_since_baseline(date: pd.Timestamp) -> float:
    """Approximate fractional months between two timestamps."""
    return (date - baseline_date).days / 30.4375

def ai_cost_model(date: pd.Timestamp) -> float:
    """Exponential decay model for inference cost."""
    return INITIAL_INFERENCE_COST * (0.5 ** (months_since_baseline(date) / HALVING_PERIOD_MONTHS))

logistic_df["ai_cost_model"] = logistic_df["release_date"].apply(ai_cost_model)

observed_generation_cost = (
    runs.groupby("alias")["generation_cost"].median().rename("median_generation_cost")
)
logistic_df = logistic_df.merge(
    observed_generation_cost,
    left_on="agent",
    right_index=True,
    how="left",
)

logistic_df["cost_cutoff_seconds"] = logistic_df["ai_cost_model"] / wage_per_second
logistic_df["cost_cutoff_minutes"] = logistic_df["cost_cutoff_seconds"] / 60
logistic_df["effective_time_minutes"] = np.minimum(
    logistic_df["p50_minutes"],
    logistic_df["cost_cutoff_minutes"],
)
logistic_df["cost_constraint_active"] = (
    logistic_df["effective_time_minutes"] < logistic_df["p50_minutes"]
)

logistic_df[[
    "agent",
    "release_date",
    "p50_minutes",
    "effective_time_minutes",
    "ai_cost_model",
    "median_generation_cost",
    "cost_constraint_active",
]]


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(
    logistic_df["release_date"],
    logistic_df["p50_minutes"],
    marker="o",
    label="Baseline 50% time horizon",
)
ax.plot(
    logistic_df["release_date"],
    logistic_df["effective_time_minutes"],
    marker="o",
    label="Cost-constrained frontier",
)

active = logistic_df["cost_constraint_active"]
if active.any():
    ax.scatter(
        logistic_df.loc[active, "release_date"],
        logistic_df.loc[active, "effective_time_minutes"],
        color="tab:red",
        label="Cost constraint binding",
        zorder=5,
    )

ax.set_yscale("log")
ax.set_ylabel("Human-equivalent task time (minutes)")
ax.set_xlabel("Model release date")
ax.set_title("Cost-effectiveness constraint on time-to-scale")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### Takeaways

The cost-adjusted frontier hugs the original time horizon until the AI's declining inference cost drops below the human labor cost for the same tasks.
Once that happens (shown in red), humans are out-competed on cost even when they can finish the task faster, so the effective frontier bends downward earlier than the pure time curve.
The parameters above can be tuned to explore alternative wage rates or hardware cost trends.
